# Solar feature segmentation
Classifies every on-disk pixel of one SDO observation into six classes. Each feature type is found the same way: first detect what stands out from the quiet Sun, then let K-means decide what it is.
- **Umbra, penumbra, pore:** dark regions in HMI Ic; K-means splits sunspots from pores, then umbra from penumbra
- **Plage, network:** bright regions in AIA 1700 Å; K-means splits plage from network
- **Quiet Sun:** everything else on the disk

## 1. Imports

In [1]:
import time
import numpy as np
import pandas as pd

import sunpy.map
from IPython.display import display

from solar_segmentation.download import download_observation, select_observation
from solar_segmentation.clv import clv_correct
from solar_segmentation.alignment import align_to_aia
from solar_segmentation.segmentation import (segment_spots, segment_bright, build_label_map, pixel_counts)
from solar_segmentation.plotting import clv_tabs, alignment_tabs, segmentation_tabs
from solar_segmentation.output import save_label_map

stage_times = {}

## 2. Configuration:
### 2.1 Settings
Set your JSOC email and the observation time.

In [2]:
JSOC_EMAIL = "youremail@hotmail.com"   # Email must be registered for JSOC exports
TIME_START = '2014-10-22T00:00:00'         # The first HMI slot with data at or after this time is used
DATA_DIR = './data'                        # Downloaded FITS files
OUTPUT_DIR = './output'                    # Saved label maps
 
# Display
DISPLAY_STEP = 2      # Plot every nth pixel
DISPLAY_SIZE = 800    # Figure size in pixels

### 2.2 Pipeline parameters
Method settings; the defaults are the values used in the paper.

In [3]:
# Download
HMI_SEARCH_HOURS = 2       # how far after TIME_START to look for an HMI slot with data
MAX_RETRIES = 3            # attempts per channel
MIN_FILE_SIZE = 1_000_000  # bytes; smaller files are treated as failed downloads

# CLV correction
B_THRESHOLD = 50   # G; quiet Sun = |B| below this, also the minimum mean |B| of a sunspot or pore
POLY_ORDER = 5     # order of the polynomial in mu

# Sunspots and pores: dark regions in smoothed Ic
SPOT_SMOOTH_SIGMA = 2   # Gaussian smoothing of Ic in pixels, washes out granulation lanes
DARK_SIGMA = 5          # dark = this many quiet-Sun standard deviations below the quiet-Sun median
MIN_SPOT_REGION = 4     # ignore dark regions smaller than this (pixels)

# Plage and network: bright regions in smoothed AIA 1700
BRIGHT_SMOOTH_SIGMA = 2   # Gaussian smoothing of AIA 1700 in pixels
BRIGHT_SIGMA = 3          # bright = this many quiet-Sun standard deviations above the quiet-Sun median
MIN_NETWORK_SIZE = 4      # min pixels per bright region (network or plage)
MIN_PLAGE_SIZE = 450      # min pixels per plage region; smaller plage-like regions count as network
KMEANS_SEED = 42          # random seed for every K-means

# Classes: label value and colour
CONTOUR_WIDTH = 1.5
SEG_CLASSES = {
    'off-disk':      (-1, '#FFFFFF'),   # white: blends into the page
    'QS':            (0,  '#E8E8E8'),   # light grey: most of the disk, so it stays quiet
    'network':       (1,  '#009E73'),   # bluish green: stands out on the red AIA 1700 map
    'plage':         (2,  '#CC79A7'),   # reddish pink: stands out on the yellow AIA 1600 map
    'penumbra':      (3,  '#F0E442'),   # yellow: bright ring around the umbra
    'umbra':         (4,  '#D55E00'),   # vermillion: strongest colour for the smallest class
    'pore':          (5,  '#0072B2'),   # blue: clearly separate from the sunspot colours
}

## 3. Download:

In [4]:
t0 = time.time()
downloaded, t_slot = download_observation(TIME_START, 
                                          JSOC_EMAIL, 
                                          DATA_DIR, 
                                          search_hours  = HMI_SEARCH_HOURS,
                                          max_retries   = MAX_RETRIES, 
                                          min_file_size = MIN_FILE_SIZE)
files, file_table = select_observation(downloaded, t_slot)
display(file_table)
stage_times['Download'] = time.time() - t0

HMI: first slot with Ic and magnetogram data is 2014.10.22_00:00:00_TAI (2014-10-21T23:59:25.000 UTC)

AIA 1700: attempt 1/3...
AIA 1700: 2 results found


2026-11-10 02:21:22 - drms - INFO: Export request pending. [id=JSOC_20261109_004509, status=2]
2026-11-10 02:21:22 - drms - INFO: Waiting for 5 seconds...
2026-11-10 02:21:28 - sunpy - INFO: 2 URLs found for download. Full request totaling 25MB


INFO: 2 URLs found for download. Full request totaling 25MB [sunpy.net.jsoc.jsoc]


Files Downloaded:   0%|          | 0/2 [00:00<?, ?file/s]

AIA 1700: SUCCESS -- 2 valid file(s)

AIA 1600: attempt 1/3...
AIA 1600: 3 results found


2026-11-10 02:21:42 - drms - INFO: Export request pending. [id=JSOC_20261109_004514, status=2]
2026-11-10 02:21:42 - drms - INFO: Waiting for 5 seconds...
2026-11-10 02:21:48 - sunpy - INFO: 3 URLs found for download. Full request totaling 30MB


... (4 outputs left out)

hmi.Ic_720s[2014.10.22_00:00:00_TAI]{continuum}: SUCCESS -- 1 valid file(s)

hmi.M_720s[2014.10.22_00:00:00_TAI]{magnetogram}: attempt 1/3...


2026-11-10 02:22:04 - drms - INFO: Export request pending. [id=JSOC_20261109_004530, status=2]
2026-11-10 02:22:04 - drms - INFO: Waiting for 5 seconds...


hmi.M_720s[2014.10.22_00:00:00_TAI]{magnetogram}: SUCCESS -- 1 valid file(s)


,Time (UTC),Gap (s),File
Channel,,,
ic,2014-10-21T23:59:25.000,0.0,hmi.ic_720s.20141022_000000_TAI.1.continuum.fits
mag,2014-10-21T23:59:25.000,0.0,hmi.m_720s.20141022_000000_TAI.1.magnetogram.fits
aia1700,2014-10-21T23:59:19.215,-5.8,aia.lev1_uv_24s.2014-10-21T235919Z.1700.image_...
aia1600,2014-10-21T23:59:29.571,4.6,aia.lev1_uv_24s.2014-10-21T235930Z.1600.image_...


## 4. Loading raw disk images as sunpy maps:

In [5]:
t0 = time.time()
aia_map     = sunpy.map.Map(files['aia1700'])
aia1600_map = sunpy.map.Map(files['aia1600'])
ic_map      = sunpy.map.Map(files['ic'])
mag_map     = sunpy.map.Map(files['mag'])
stage_times['Making sunpy maps'] = time.time() - t0

## 5. Rotation & reprojection:

In [6]:
t0 = time.time()
aia_rotated, aia1600_reprojected, ic_reprojected, mag_reprojected = align_to_aia(
    aia_map, aia1600_map, ic_map, mag_map)

display(alignment_tabs(aia_rotated, aia1600_reprojected, ic_reprojected, mag_reprojected,
                       step=DISPLAY_STEP, size=DISPLAY_SIZE))
stage_times['Rotate + reproject'] = time.time() - t0

2026-11-10 02:22:18 - reproject.common - INFO: Calling _reproject_full in non-dask mode


Rotated AIA shape: (4098, 4098)


2026-11-10 02:23:04 - reproject.common - INFO: Calling _reproject_full in non-dask mode
2026-11-10 02:23:50 - reproject.common - INFO: Calling _reproject_full in non-dask mode


## 6. Limb darkening removal:

In [7]:
t0 = time.time()
clv_kw = dict(b_threshold=B_THRESHOLD, poly_order=POLY_ORDER)
ic_corrected, _, _ = clv_correct(aia_rotated, ic_reprojected, mag_reprojected, **clv_kw)
aia_corrected, _, _ = clv_correct(aia_rotated, aia_rotated.data, mag_reprojected, **clv_kw)
aia1600_corrected, _, _ = clv_correct(aia_rotated, aia1600_reprojected, mag_reprojected, **clv_kw)

display(clv_tabs({'HMI Ic':   (ic_reprojected, ic_corrected, 'gray'),
                  'AIA 1700': (aia_rotated.data, aia_corrected, 'sdoaia1700'),
                  'AIA 1600': (aia1600_reprojected, aia1600_corrected, 'sdoaia1600')},
                 b_threshold=B_THRESHOLD, step=DISPLAY_STEP, size=DISPLAY_SIZE))
stage_times['CLV correction'] = time.time() - t0

## 7. Segmentation:

In [8]:
t0 = time.time()
on_disk = (np.isfinite(ic_corrected) & np.isfinite(mag_reprojected)
           & np.isfinite(aia_corrected) & np.isfinite(aia1600_corrected))

# Umbra, penumbra, pores: dark regions in Ic, then K-means
umbra, penumbra, pore, spot_regions = segment_spots(ic_corrected, mag_reprojected, on_disk,
                                                    smooth_sigma=SPOT_SMOOTH_SIGMA, dark_sigma=DARK_SIGMA,
                                                    min_region=MIN_SPOT_REGION, min_b=B_THRESHOLD,
                                                    seed=KMEANS_SEED)
display(spot_regions.head(20).round(3))
non_spot = on_disk & ~(umbra | penumbra | pore)

# Plage and network: bright regions in AIA 1700, then K-means
plage, network, bright_regions = segment_bright(aia_corrected, aia1600_corrected, mag_reprojected, non_spot,
                                                smooth_sigma=BRIGHT_SMOOTH_SIGMA, bright_sigma=BRIGHT_SIGMA,
                                                min_network_size=MIN_NETWORK_SIZE, min_plage_size=MIN_PLAGE_SIZE,
                                                seed=KMEANS_SEED)
display(bright_regions.sort_values('Area (px)', ascending=False).head(15).round(2))

seg_map, masks = build_label_map(on_disk, network, plage, pore, penumbra, umbra, SEG_CLASSES)
display(pixel_counts(seg_map, SEG_CLASSES).round(2))
stage_times['Segmentation'] = time.time() - t0

Quiet Sun Ic: median 1.000, sigma 0.0100 -> dark below 0.950
70 dark regions of at least 4 px with mean |B| >= 50 G
Umbra/penumbra Ic centres: [0.343 0.782]


,Area (px),Mean Ic,Min Ic,Mean |B| (G),Class
Region,,,,,
8,58956.0,0.724,0.052,538.693,sunspot
58,2326.0,0.725,0.197,482.619,sunspot
69,2304.0,0.774,0.336,538.406,sunspot
46,1405.0,0.761,0.247,437.534,sunspot
6,960.0,0.852,0.556,325.869,sunspot
7,538.0,0.865,0.541,338.311,sunspot
28,414.0,0.888,0.573,658.155,sunspot
53,362.0,0.856,0.559,422.923,sunspot
76,236.0,0.868,0.636,198.269,sunspot


Quiet Sun 1700: median 0.982, sigma 0.1266 -> bright above 1.362
3519 bright regions: 135 plage, 3384 network


,Area (px),Mean AIA 1600,Mean |B| (G),Class
Region,,,,
566,68766.0,2.11,166.19,plage
620,46547.0,2.10,117.08,plage
1087,12898.0,2.38,102.86,plage
1695,10645.0,2.06,142.47,plage
954,9471.0,2.48,70.84,plage
2406,9074.0,1.96,119.51,plage
1098,7353.0,2.01,116.04,plage
1318,5049.0,1.87,93.07,plage
2325,5025.0,2.03,134.64,plage


,Label,Pixels,% of disk
Class,,,
QS,0,7196969,92.58
network,1,197123,2.54
plage,2,314030,4.04
penumbra,3,55264,0.71
umbra,4,9260,0.12
pore,5,889,0.01
TOTAL,-,7773535,100.00


## 8. Plotting segmentation:

In [9]:
t0 = time.time()
display(segmentation_tabs(seg_map, ic_corrected, aia_corrected, aia1600_corrected, masks,
                          seg_classes=SEG_CLASSES, contour_width=CONTOUR_WIDTH,
                          step=DISPLAY_STEP, size=DISPLAY_SIZE))
stage_times['Plotting'] = time.time() - t0

    'data': [{'dx': np.int64(2),
              'dy': np.int64(2),
…

## 9. Saving segmented map as a FITS file:

In [10]:
t0 = time.time()
params = dict(TIME_START  = TIME_START, 
              B_THRESHOLD = B_THRESHOLD, 
              POLY_ORDER  = POLY_ORDER,
              SPOT_SMOOTH_SIGMA = SPOT_SMOOTH_SIGMA, 
              DARK_SIGMA  = DARK_SIGMA, 
              MIN_SPOT_REGION = MIN_SPOT_REGION,
              BRIGHT_SMOOTH_SIGMA = BRIGHT_SMOOTH_SIGMA, 
              BRIGHT_SIGMA = BRIGHT_SIGMA,
              MIN_NETWORK_SIZE  = MIN_NETWORK_SIZE, 
              MIN_PLAGE_SIZE = MIN_PLAGE_SIZE, 
              KMEANS_SEED    = KMEANS_SEED)

out_path = f"{OUTPUT_DIR}/segmentation_{t_slot.strftime('%Y%m%d_%H%M%S')}.fits"
save_label_map(out_path, seg_map, aia_rotated, SEG_CLASSES, files, params)
stage_times['Saving'] = time.time() - t0

Saved ./output/segmentation_20141021_235925.fits


## 10. Timer summary table:

In [11]:
timings = pd.DataFrame({'Time (s)': stage_times}).rename_axis('Stage')
timings['% of total'] = 100 * timings['Time (s)'] / timings['Time (s)'].sum()
timings.loc['TOTAL'] = [timings['Time (s)'].sum(), 100.0]
display(timings.round(2))

,Time (s),% of total
Stage,,
Download,63.45,22.12
Making sunpy maps,2.62,0.91
Rotate + reproject,159.74,55.68
CLV correction,29.66,10.34
Segmentation,16.25,5.67
Plotting,15.04,5.24
Saving,0.14,0.05
TOTAL,286.90,100.00
